# 05 · Partitioning

> **Module 4 · Partitioning & Clustering · Lectures 4.1–4.5**

In this notebook you will:

- Create ingestion-time and date-column partitioned tables
- Prove partition pruning with a byte comparison
- Use partition decorators and `REPLACE PARTITION`
- Set `require_partition_filter` and `partition_expiration_days`
- Understand the 10,000-partition limit

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. The three partitioning types

| Type | Pseudo/column | Use when |
|------|---------------|----------|
| **Ingestion-time** | `_PARTITIONDATE` | The source has no trustworthy event date |
| **Date column** | Your own `DATE`/`TIMESTAMP` column | You filter by business date |
| **Integer range** | `RANGE_BUCKET` | You partition on a monotonically increasing integer |

In [ ]:
# Ingestion-time partitioned
ingest = bigquery.Table(f"{PROJECT_ID}.raw.events_ingest", schema=[
    bigquery.SchemaField("event_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING"),
    bigquery.SchemaField("event_type", "STRING"),
    bigquery.SchemaField("event_ts", "TIMESTAMP"),
    bigquery.SchemaField("payload", "JSON"),
])
ingest.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    expiration_ms=90 * 24 * 60 * 60 * 1000,
)
ingest.description = "Raw events partitioned by ingestion date"
ingest = client.create_table(ingest, exists_ok=True)
print(f"✓ {ingest.full_table_id}")
print(f"  partitioning: {ingest.time_partitioning.type_} (ingestion time)")

In [ ]:
# Date-column partitioned
date_table = bigquery.Table(f"{PROJECT_ID}.curated.orders_part", schema=[
    bigquery.SchemaField("order_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("total_amount", "NUMERIC"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("region", "STRING"),
])
date_table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="order_date",
    expiration_ms=730 * 24 * 60 * 60 * 1000,
    require_partition_filter=True,
)
date_table.clustering_fields = ["customer_id", "region"]
date_table = client.create_table(date_table, exists_ok=True)
print(f"✓ {date_table.full_table_id}")
print(f"  partitioning: {date_table.time_partitioning.field}")
print(f"  clustering:  {date_table.clustering_fields}")

In [ ]:
# Integer-range partitioned (via SQL — no direct SDK attribute)
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.events_int_range` (
      event_id   STRING,
      event_ts   TIMESTAMP,
      user_id    STRING,
      event_type STRING
    )
    PARTITION BY RANGE_BUCKET(event_ts, GENERATE_ARRAY_ARRAY(
      TIMESTAMP '2026-01-01 00:00:00',
      TIMESTAMP '2027-01-01 00:00:00',
      INTERVAL 1 MONTH
    ))
    """
)
print("✓ Integer-range partitioned table created")

## 2. Load data into partitions

Write to a **specific partition** with a decorator (`table$20260115`) to make daily
loads idempotent.

In [ ]:
import random
from datetime import date, timedelta

random.seed(7)
REGIONS = ["EMEA", "AMER", "APAC", "LATAM"]
STATUSES = ["active", "complete", "cancelled"]

rows = []
for day_offset in range(3):
    d = date(2026, 1, 15) - timedelta(days=day_offset)
    partition = d.strftime("%Y%m%d")
    day_rows = []
    for i in range(200):
        day_rows.append(
            {
                "order_id": f"o{partition}_{i:05d}",
                "customer_id": f"c{random.randint(1, 50):04d}",
                "order_date": d.isoformat(),
                "total_amount": round(random.uniform(10, 500), 2),
                "status": random.choice(STATUSES),
                "region": random.choice(REGIONS),
            }
        )
    errors = client.insert_rows_json(
        client.dataset("curated").table(f"orders_part${partition}"), day_rows
    )
    print(f"  {partition}: {len(day_rows)} rows, errors={len(errors)}")

In [ ]:
# Inspect the partitions
df = run_to_dataframe(
    f"""
    SELECT
      partition_id,
      total_rows,
      ROUND(total_logical_bytes / 1024, 1) AS kb
    FROM `{PROJECT_ID}.curated.orders_part.__PARTITIONS__`
    ORDER BY partition_id DESC
    """
)
display(df)

## 3. Prove partition pruning

This is the single most important measurement in BigQuery. Compare the bytes a
full scan costs against a pruned scan.

In [ ]:
FULL_SCAN = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part`"
PRUNED = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part` WHERE order_date = DATE '2026-01-15'"
DECORATOR = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part$20260115`"
FILTERED_COLS = f"SELECT order_id FROM `{PROJECT_ID}.curated.orders_part` WHERE order_date = DATE '2026-01-15'"

print(f"{'variant':<28} {'bytes':>14} {'TiB':>9} {'est. cost':>11}")
print("-" * 66)
for label, sql in [
    ("full scan (no filter)", FULL_SCAN),
    ("pruned (WHERE on column)", PRUNED),
    ("pruned (partition decorator)", DECORATOR),
    ("pruned + specific columns", FILTERED_COLS),
]:
    try:
        est = estimate_cost(sql)
        print(f"{label:<28} {est['bytes_processed']:>14,} {est['tib']:>9.4f} ${est['estimated_cost_usd']:>10.6f}")
    except Exception as exc:
        print(f"{label:<28} ERROR: {str(exc)[:60]}")

In [ ]:
# Why a function on the partition column defeats pruning
BAD = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part` WHERE DATE_ADD(order_date, INTERVAL 0 DAY) = DATE '2026-01-15'"
GOOD = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part` WHERE order_date = DATE '2026-01-15'"

print("Filter on a function of the partition column (no pruning):")
print(f"  {estimate_cost(BAD)['bytes_processed']:,} bytes")
print("\nFilter on the column directly (prunes):")
print(f"  {estimate_cost(GOOD)['bytes_processed']:,} bytes")

In [ ]:
# The half-open range pattern — the most efficient form
HALF_OPEN = f"""
SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part`
WHERE order_date >= DATE '2026-01-14' AND order_date < DATE '2026-01-16'
"""

CLOSED = f"""
SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part`
WHERE order_date BETWEEN DATE '2026-01-14' AND DATE '2026-01-15'
"""

print(f"half-open range:  {estimate_cost(HALF_OPEN)['bytes_processed']:,} bytes")
print(f"BETWEEN:         {estimate_cost(CLOSED)['bytes_processed']:,} bytes")
print("\nBoth prune equally — half-open is just safer for timestamps.")

## 4. `require_partition_filter`

Turns the most common BigQuery cost accident into a compile-time error.

In [ ]:
table = client.get_table(f"{PROJECT_ID}.curated.orders_part")
print(f"require_partition_filter: {table.time_partitioning.require_partition_filter}")

# Try to run an unfiltered query — it should be rejected
try:
    run_query(FULL_SCAN)
    print("\n⚠️  Query succeeded — the filter requirement is not enforced")
except Exception as exc:
    print(f"\n✅ Blocked as intended:\n   {str(exc)[:200]}")

## 5. Partition decorators and `REPLACE PARTITION`

A partition decorator addresses a partition directly. `REPLACE PARTITION` is the
cheap way to rewrite one partition's contents.

In [ ]:
df = run_to_dataframe(
    f"""
    SELECT
      (SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part`)        AS all_rows,
      (SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part$20260115`) AS one_partition,
      (SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_part$20260101:20260131`) AS january
    """
)
display(df)

In [ ]:
# REPLACE PARTITION — change every row in a partition in one statement
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.orders_part$20260115` AS
    SELECT * EXCEPT(status), 'reconciled' AS status
    FROM `{PROJECT_ID}.curated.orders_part$20260115`
    """
)

df = run_to_dataframe(
    f"""
    SELECT status, COUNT(*) AS n
    FROM `{PROJECT_ID}.curated.orders_part$20260115`
    GROUP BY status
    """
)
display(df)

In [ ]:
# Compare: UPDATE (rewrites the table) vs REPLACE PARTITION (rewrites one partition)
print("REPLACE PARTITION: rewrites one partition — cost is proportional to that partition")
print("UPDATE:            rewrites every affected block, with per-row DML overhead")
print("Rule of thumb:     if you change more than ~10% of a partition, replace the partition.\n")

# Add a partition before inserting into a specific one
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_part` ADD PARTITION IF NOT EXISTS DATE '20260116'"
)
print("✓ Added partition 20260116")

## 6. Partition expiration — the cheapest cost control

Set it on the table, and it removes old partitions automatically.

In [ ]:
# Change the expiration
table = client.get_table(f"{PROJECT_ID}.raw.events_ingest")
table.time_partitioning.expiration_ms = 30 * 24 * 60 * 60 * 1000
table = client.update_table(table, update_mode="update_fields")
print(f"✓ expiration now: {round(table.time_partitioning.expiration_ms / 86_400_000)} days")

# Or with SQL
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_part` "
    "SET OPTIONS (partition_expiration_days = 365)"
)
print("✓ orders_part now expires partitions after 365 days")

In [ ]:
# A dataset-level default for new tables
dataset = client.get_dataset(f"{PROJECT_ID}.scratch")
dataset.default_table_expiration_ms = 7 * 24 * 60 * 60 * 1000
dataset = client.update_dataset(dataset, update_mode="update_fields")
print(f"✓ scratch: new tables auto-expire after 7 days")

## 7. The 10,000-partition limit

| Granularity | Partitions per year | Years before the limit |
|-------------|--------------------|------------------------|
| Hourly | 8,760 | ~1.1 years |
| Daily | 365 | ~27 years |
| Monthly | 12 | ~833 years |

**If you need hourly granularity at scale:** partition daily and cluster on the hour
column. You keep 10 years of hourly detail without hitting the limit.

In [ ]:
# Count the partitions you have
df = run_to_dataframe(
    f"""
    SELECT COUNT(*) AS partition_count
    FROM `{PROJECT_ID}.curated.orders_part.__PARTITIONS__`
    """
)
display(df)

# The daily + clustered hour pattern
hourly = bigquery.Table(f"{PROJECT_ID}.curated.events_hourly", schema=[
    bigquery.SchemaField("event_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("event_ts", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING"),
    bigquery.SchemaField("event_type", "STRING"),
])
hourly.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="event_ts",
    expiration_ms=90 * 24 * 60 * 60 * 1000,
)
hourly.clustering_fields = ["event_type", "user_id"]
hourly = client.create_table(hourly, exists_ok=True)
print(f"✓ {hourly.full_table_id} — daily partitions, clustered by event_type")

## 8. Partitioning and DML — the `MERGE` rule

A BigQuery `MERGE` **must** match on the partition key. Without it, the merge
scans the whole table.

In [ ]:
MERGE_SQL = f"""
MERGE `{PROJECT_ID}.curated.orders_part` AS t
USING (
  SELECT 'o20260115_00001' AS order_id, 'c0001' AS customer_id,
         DATE '2026-01-15' AS order_date, 999.99 AS total_amount,
         'complete' AS status, 'EMEA' AS region
) AS s
ON t.order_id = s.order_id AND t.order_date = s.order_date
WHEN MATCHED THEN UPDATE SET t.total_amount = s.total_amount, t.status = s.status
WHEN NOT MATCHED THEN INSERT
  (order_id, customer_id, order_date, total_amount, status, region)
VALUES (s.order_id, s.customer_id, s.order_date, s.total_amount, s.status, s.region)
"""

job = client.query(MERGE_SQL)
job.result()
print(f"✓ MERGE affected {job.num_dml_affected_rows} rows")

df = run_to_dataframe(
    f"""
    SELECT order_id, total_amount, status
    FROM `{PROJECT_ID}.curated.orders_part`
    WHERE order_date = DATE '2026-01-15' AND order_id = 'o20260115_00001'
    """
)
display(df)

## 9. Partition maintenance

Copy, restore, and swap partitions.

In [ ]:
# Copy a partition to a backup table
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.backup.orders_20260115` AS
    SELECT * FROM `{PROJECT_ID}.curated.orders_part$20260115`
    """
)
print("✓ Copied partition to backup")

# Restore it
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.orders_part$20260115` AS
    SELECT * FROM `{PROJECT_ID}.backup.orders_20260115`
    """
)
print("✓ Restored partition")

# Atomic swap — build, then replace in one statement
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.staging.orders_new$20260115` AS
    SELECT * FROM `{PROJECT_ID}.curated.orders_part$20260115`
    """
)
run_query(
    f"""
    CREATE OR REPLACE TABLE `{PROJECT_ID}.curated.orders_part$20260115` AS
    SELECT * FROM `{PROJECT_ID}.staging.orders_new$20260115`
    """
)
print("✓ Swapped partition")

## 🔑 Takeaways

- Three types: **ingestion-time**, **date column**, **integer range**.
- **Partition pruning is the biggest cost lever** — always filter the partition column
  directly, never a function of it.
- `require_partition_filter = TRUE` turns a costly scan into an error.
- `partition_expiration_days` is the cheapest cost control in the platform.
- **`REPLACE PARTITION` beats `UPDATE`** for bulk changes inside a partition.
- **`MERGE` must match on the partition key.**
- 10,000 partitions per table — daily + clustered hour for hourly detail at scale.

---

## 📚 Next

[06 · Clustering](./06-clustering.ipynb)